# Investor-State Dispute Settlement System as a Barrier to Environmental Justice

## Supplementary Data Analysis Notebook

### Associated Publication

**Title:** Investor-State Dispute Settlement System as a Barrier to Environmental Justice

**Research Author(s):**
- Godwin, E.L.

**Journal:** Journal Name (Year)

**DOI:** xx.xxxx/xxxxx

### Notebook Author

**Name:** Bolton, K.P.

**Role:** Data collection, processing & analysis, visualisation, and computational reproducibility

### Purpose

This notebook reproduces and extends the analyses described in the associated publication. It contains data preparation, statistical analyses, figure generation, and supporting computational workflows.

In [2]:
# Add project root to sys.path for imports
import sys
from pathlib import Path

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))

### ICSID Worldbank data


Only use the cell below to download data if needed.  Cell keeped for transparency, reproducibility and reference.

In [2]:
## Download all ICSID cases from url.  Also retrive only mining cases (es = 101)

# import requests
# import pandas as pd

# url = "https://icsid.worldbank.org/api/all/cases"

# params = {
#     "es": 101
# }

# r = requests.get(url)
# r_mining = requests.get(url, params=params)

# data = r.json()
# data_mining = r_mining.json()

# cases = data["data"]["GetAllCasesResult"]
# cases_mining = data_mining["data"]["GetAllCasesResult"]

# df = pd.json_normalize(cases)
# df_mining = pd.json_normalize(cases_mining)

## Save raw dataframes to csv files for documentation

# df.to_csv(r"../data/raw/icsid_cases_all_202605.csv", index=False)
#df_mining.to_csv(r"../data/raw/icsid_cases_mining_202605.csv", index=False)


In [3]:
import pandas as pd
df_raw = pd.read_csv(r"../data/raw/icsid_cases_all_202605.csv")
df = df_raw.copy()

df_mining_raw = pd.read_csv(r"../data/raw/icsid_cases_mining_202605.csv")
df_mining = df_mining_raw.copy()

Columns `casedecisions` and `caseproceedings` are in json object format.  Extract all the available data from these objects into a usuable dataframe.

In [4]:
# Extract data from the case proceeding and case decision dataframes and join to the main dataframe

import ast

df["casedecisions"] = df["casedecisions"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else None)
df["caseproceedings"] = df["caseproceedings"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else None)

case_decisions_col = pd.json_normalize(df["casedecisions"].explode()).reset_index(drop=True)
case_proceedings_col = pd.json_normalize(df["caseproceedings"].explode()).reset_index(drop=True)
    
df_expanded = pd.concat([df.drop(columns=["casedecisions", "caseproceedings"]), case_decisions_col, case_proceedings_col], axis=1)

# Clean up the large dataset by dropping columns with all null values and columns with all empty strings. Also, join the case proceeding and case decision data to the main dataframe.
df_clean = df_expanded.loc[:,
    ~(
        df_expanded.replace('',pd.NA)
        .isna()
        .all()
    )]

In [5]:
# Extract the messy/raw data of the country name from the `respondent` column and standardise it using the pycountry library.
from utils.country_utils import extract_country
df_clean[["country_clean", "country_iso3", "extraction_method"]] = df_clean["respondent"].apply(extract_country)

In [6]:
print(f"Total number of cases: {df_clean.shape[0]}")
print(f"Total number of cases without a country name: {df_clean['country_clean'].isna().sum()}")


# Code was used to find save the cases that could not be found using the extract_country method.
# # Save dataframe of cases without a country name
# df_cases_without_state = pd.Series(df_clean[df_clean["country_clean"].isna()][["caseno", "respondent"]].drop_duplicates().values.tolist())
# df_cases_without_state.to_csv(r"../data/interim/cases_without_state_icsid.csv", index=True) # Only uncomment if one wishes to save the interim dataset again.

Total number of cases: 1140
Total number of cases without a country name: 26


In [7]:
missing_countries = df_clean.loc[df_clean["country_clean"].isna()]

missing_cases = sorted(
    missing_countries["caseno"]
    .dropna()
    .unique()
)

print("=" * 60)
print("MISSING COUNTRY ENTRIES")
print("=" * 60)

if len(missing_cases) == 0:
    print("✓ No missing date entries found.")
else:
    print(f"Total number of cases missing a country: {len(missing_cases)}")
    for case in missing_cases:
        print(f"• {case}")

MISSING COUNTRY ENTRIES
Total number of cases missing a country: 26
• ARB/01/10
• ARB/05/12
• ARB/06/21
• ARB/07/3
• ARB/08/10
• ARB/10/11
• ARB/10/18
• ARB/10/20
• ARB/12/28
• ARB/13/24
• ARB/14/35
• ARB/19/18
• ARB/19/3
• ARB/20/50
• ARB/22/19
• ARB/22/4
• ARB/24/39
• ARB/25/12
• ARB/25/43
• ARB/25/44
• ARB/26/5
• ARB/76/1
• ARB/92/2
• ARB/98/8
• CONC(AF)/12/2
• CONC/19/1


In [8]:
# Manually add in the respondent countries per case.  Independently looked up.
from utils.constant_variables import ICSID_MANUAL_COUNTRY_CLEANUP

# Find rows in dataframe that need to be updated
mask = (
    df_clean["country_clean"].isna()
    &
    df_clean["caseno"]
    .isin(ICSID_MANUAL_COUNTRY_CLEANUP.keys())
)

# Update only missing dates
df_clean.loc[
    mask,
    "country_clean"
] = (
    df_clean.loc[
        mask,
        "caseno"
    ].map(ICSID_MANUAL_COUNTRY_CLEANUP)
)

# All cases have an associated country
df_clean["country_clean"].isna().sum()

np.int64(0)

In [9]:
# Convert dates into datetime objects and check to see if there are any dud values.
df_clean["dateregistered"] = pd.to_datetime(df_clean["dateregistered"], errors="coerce")

parsed_dates = pd.to_datetime(
    df_clean["dateregistered"],
    format="%Y-%m-%d",
    errors="coerce"
)

bad_mask = (
    parsed_dates.isna() &
    df_clean["dateregistered"].notna()
)

missing_dates = df_clean.loc[df_clean["dateregistered"].isna()]
problem_rows = df_clean.loc[bad_mask][["caseno", "dateregistered"]]

C:\Users\User\AppData\Local\Temp\ipykernel_4748\2985750509.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["dateregistered"] = pd.to_datetime(df_clean["dateregistered"], errors="coerce")


In [10]:
missing_cases = sorted(
    missing_dates["caseno"]
    .dropna()
    .unique()
)

problem_cases = sorted(
    problem_rows["caseno"]
    .dropna()
    .unique()
)

print("=" * 60)
print("MISSING DATE ENTRIES")
print("=" * 60)

if len(missing_cases) == 0:
    print("✓ No missing date entries found.")
else:
    for case in missing_cases:
        print(f"• {case}")

print("\n")

print("=" * 60)
print("UNPARSEABLE DATE ENTRIES")
print("=" * 60)

if len(problem_cases) == 0:
    print("✓ No unparseable date entries found.")
else:
    for case in problem_cases:
        print(f"• {case}")

MISSING DATE ENTRIES
• ADM/21/1
• UNCT/23/4


UNPARSEABLE DATE ENTRIES
✓ No unparseable date entries found.


Two cases in the data do not contain an entry for `dateregistered`.  Manually enter the date with reference to sources.

UNCT/23/4 - 30/11/2023 - https://icsid.worldbank.org/cases/case-database/case-detail?CaseNo=UNCT/23/4 

ADM/21/1 - 07/08/2020 - https://jusmundi.com/en/document/decision/en-asiaphos-limited-v-peoples-republic-of-china-composition-of-the-tribunal

In [11]:
# Create manual dictionary of cases needed to be cleaned up
from utils.constant_variables import ICSID_MANUAL_DATE_CLEANUP

# Find rows in dataframe that need to be updated
mask = (
    df_clean["dateregistered"].isna()
    &
    df_clean["caseno"]
    .isin(ICSID_MANUAL_DATE_CLEANUP.keys())
)

# Update only missing dates
df_clean.loc[
    mask,
    "dateregistered"
] = (
    df_clean.loc[
        mask,
        "caseno"
    ].map(ICSID_MANUAL_DATE_CLEANUP)
)

# Convert to datetime
df_clean["dateregistered"] = pd.to_datetime(
    df_clean["dateregistered"],
    errors = "coerce"
)

In [12]:
# Verify that cases have been cleaned up
df_clean.loc[df_clean["caseno"].isin(ICSID_MANUAL_DATE_CLEANUP.keys())][["caseno", "dateregistered"]]

,caseno,dateregistered
142,UNCT/23/4,2023-11-30
202,ADM/21/1,2020-08-07


In [13]:
# Extract the day, month, and year from the data column of the dataset
df_clean["dateregistered"] = pd.to_datetime(df_clean["dateregistered"], errors="coerce")
df_clean["registration_year"] = df_clean["dateregistered"].dt.year.astype("Int64")
df_clean["registration_month"] = df_clean["dateregistered"].dt.month.astype("Int64")
df_clean["registration_day"] = df_clean["dateregistered"].dt.day.astype("Int64")

In [14]:
# Using the extracted filtered dataframe df_mining, mark which cases relate to the mining sector in the working dataframe
mining_cases = df_mining["caseno"]
df_clean["miningcase"] = df_clean["caseno"].isin(mining_cases).map({True: "Yes", False: "No"})

In [15]:
# Save cleaned dataframe to csv file for documentation
df_clean.to_csv(r"../data/interim/icsid_clean_cases_202606_processed.csv")

### UNCTAD data

In [16]:
# Extract all data up to and including 2023 from UNCTAD dataset
UNCTAD_data_2023_raw = pd.read_excel(r"../data/interim/UNCTAD-ISDS-Navigator-data-set-31December2023_urlExtract.xlsx", skiprows=11, sheet_name="Source - UNCTAD ISDS Navigator")
# Extract all data from after 2023 - these values were manually extracted from the website and added to the original dataset in a new sheet
UNCTAD_data_2024_plus_raw = pd.read_excel(r"../data/interim/UNCTAD-ISDS-Navigator-data-set-31December2023_urlExtract.xlsx", sheet_name="Sheet3")

UNCTAD_data_2023 = UNCTAD_data_2023_raw.copy()
UNCTAD_data_2024_plus = UNCTAD_data_2024_plus_raw.copy()

In [17]:
# Extract all the unique ICSID case numbers from the 'FULL CASE NAME' column using regex
pattern = r'(ARB(?:\(AF\))?/\d+/\d+)'
UNCTAD_data_2023['CASE NO.'] = UNCTAD_data_2023['FULL CASE NAME'].str.extract(pattern)

# Combine the datasets into 1
UNCTAD_FULL = pd.concat([UNCTAD_data_2023, UNCTAD_data_2024_plus])

# Verify concatenation
print(f"Shape of UNCTAD_data_2024_plus: {UNCTAD_data_2024_plus.shape}")
print(f"Shape of UNCTAD_data_2023: {UNCTAD_data_2023.shape}")
print(f"Shape of UNCTAD_FULL: {UNCTAD_FULL.shape}")

Shape of UNCTAD_data_2024_plus: (118, 11)
Shape of UNCTAD_data_2023: (1332, 29)
Shape of UNCTAD_FULL: (1450, 31)


In [18]:
# Extract country from "RESPONDENT STATE" column
from utils.country_utils import extract_country
UNCTAD_FULL[["country_clean", "country_iso3", "extraction_method"]] = UNCTAD_FULL["RESPONDENT STATE"].apply(extract_country)

In [19]:
# Check that all countries for the dataset have been found
print(f"Total number of entries missing a country: {UNCTAD_FULL["country_clean"].isna().sum()}")

## Save entries missing a country in a separate dataset.  Respondent country is present in "SHORT NAME" column.  Manually update this and add into the dataset.
# UNCTAD_FULL[UNCTAD_FULL["country_clean"].isna()].to_csv("../data/interim/UNCTAD_UNMATCHED_COUNTRIES.csv", index=False)

# Read in data
UNCTAD_MANUAL_MATCH = pd.read_csv("../data/interim/UNCTAD_UNMATCHED_COUNTRIES.csv")

Total number of entries missing a country: 118


In [20]:
# Find rows where we could not extract a country and manually match them based on the 'SHORT CASE NAME' and 'RESPONDENT STATE' columns. Create a mapping dictionary to update the main dataframe.
MANUAL_MAP = dict(
    zip(
        UNCTAD_MANUAL_MATCH["SHORT CASE NAME"],
        UNCTAD_MANUAL_MATCH["RESPONDENT STATE"]
    )
)   

MASK = UNCTAD_FULL["country_clean"].isna()

UNCTAD_FULL.loc[MASK, "country_clean"] = UNCTAD_FULL.loc[MASK, "SHORT CASE NAME"].map(MANUAL_MAP)
UNCTAD_FULL.loc[MASK, "extraction_method"] = "manual"

# All cases now have a country assigned to them
print(f"Total number of entries missing a country: {UNCTAD_FULL["country_clean"].isna().sum()}")

Total number of entries missing a country: 0


In [21]:
# In the UNCTAD dataset, all entries related to mining will be referenced as "Mining and Quarrying" in the "ECONOMIC SECTOR" column.  Filter the column to find all these entries and mark it in the working dataset
UNCTAD_FULL["MINING CASE"] = (
    UNCTAD_FULL["ECONOMIC SECTOR"]
    .str.contains(
        r"mining and quarrying",
        case=False,
        na=False
    )
    .map({True: "Yes", False: "No"})
)

print(f"Total number of mining and non-mining cases in the UNCTAD data: \n{UNCTAD_FULL["MINING CASE"].value_counts()}")

Total number of mining and non-mining cases in the UNCTAD data: 
MINING CASE
No     1196
Yes     254
Name: count, dtype: int64


In [22]:
# Using the "CASE NO." column, see what entries are already present in the ICSID data.
UNCTAD_FULL["IN ICSID DATA"] = UNCTAD_FULL["CASE NO."].isin(df_clean["caseno"]).map({True: "Yes", False: "No"})
print(f"Total number of overlapping and non-overlapping cases in the UNCTAD data: \n{UNCTAD_FULL["IN ICSID DATA"].value_counts()}")

Total number of overlapping and non-overlapping cases in the UNCTAD data: 
IN ICSID DATA
Yes    791
No     659
Name: count, dtype: int64


In [23]:
# Retrive the remaining PCA case numbers and update the dataset for the next section's data clean up.
UNCTAD_FULL["CASE NO."] = UNCTAD_FULL["CASE NO."].fillna(
    UNCTAD_FULL["Notes"].str.extract(
        r"PCA Case No\.\s*([0-9]+-[0-9]+)",
        expand=False
    )
)

### PCA Data Analysis
Investor-state arbitration cases from https://pca-cpa.org/en/cases/ were manually added into an excel sheet to be ingested for analysis.

In [24]:
PCA_raw = pd.read_excel(r"../data/raw/PCA_INVESTOR_STATE_CASES_052026.xlsx")
PCA_DATA = PCA_raw.copy()

# Working with only a subset of interested columns

PCA_DATA = PCA_DATA[["Name(s) of Respondent(s)", "Case number", "Date of commencement of proceeding", "Subject matter or economic sector"]]

In [25]:
# Review any cases without a date
PCA_DATA[PCA_DATA["Date of commencement of proceeding"].isna()]

,Name(s) of Respondent(s),Case number,Date of commencement of proceeding,Subject matter or economic sector
24,The United States of America (State),2023-03,NaN,- Other -


In [26]:
# Given format of PCA cases using the year in their case numbers, will manually add this into the data.
from utils.constant_variables import PCA_MANUAL_DATE_CLEANUP

# Find rows in dataframe that need to be updated
mask = (
    PCA_DATA["Date of commencement of proceeding"].isna()
    &
    PCA_DATA["Case number"]
    .isin(PCA_MANUAL_DATE_CLEANUP.keys())
)

# Update only missing dates
PCA_DATA.loc[
    mask,
    "Date of commencement of proceeding"
] = (
    PCA_DATA.loc[
        mask,
        "Case number"
    ].map(PCA_MANUAL_DATE_CLEANUP)
)

# Convert to datetime
PCA_DATA["Date of commencement of proceeding"] = pd.to_datetime(
    PCA_DATA["Date of commencement of proceeding"],
    errors = "coerce",
    format = "mixed"
)

# Verify date cleanup
PCA_DATA[PCA_DATA["Date of commencement of proceeding"].isna()] # No missing entries.

,Name(s) of Respondent(s),Case number,Date of commencement of proceeding,Subject matter or economic sector


In [27]:
# Retrieve year for each case
PCA_DATA["Date of commencement of proceeding"] = pd.to_datetime(PCA_DATA["Date of commencement of proceeding"], errors="coerce")
PCA_DATA["year"] = PCA_DATA["Date of commencement of proceeding"].dt.year.astype("Int64")

In [28]:
# Find all PCA mining cases
PCA_DATA["MINING CASE"] = PCA_DATA["Subject matter or economic sector"].str.contains("Mining and quarrying", case=False, na=False).map({True: "Yes", False: "No"})
print(f"Total number of mining cases in the PCA dataset: {PCA_DATA["MINING CASE"].value_counts()}") # 9 mining cases


Total number of mining cases in the PCA dataset: MINING CASE
No     43
Yes     9
Name: count, dtype: int64


In [29]:
# Find all unique PCA cases not in the UNCTAD dataset
PCA_DATA["IN UNCTAD"] = PCA_DATA["Case number"].isin(UNCTAD_FULL["CASE NO."]).map({True: "Yes", False: "No"})
print(f"Total number of unique cases in PCA dataset: {PCA_DATA["IN UNCTAD"].value_counts()}")  # 40 unique cases

Total number of unique cases in PCA dataset: IN UNCTAD
No     40
Yes    12
Name: count, dtype: int64


In [30]:
# Retrieve country for each case
from utils.country_utils import extract_country
PCA_DATA[["country_clean", "country_iso3", "extraction_method"]] = PCA_DATA["Name(s) of Respondent(s)"].apply(extract_country)

# Verify all countries found
print(f"Total countries missing: {PCA_DATA["country_clean"].isna().sum()}")

Total countries missing: 0


## Creating the dataset for graphing

In [31]:
# Standardise column names for each dataset ready to merge

ICSID_MAPPING = {
    "country_clean": "country_clean",
    "caseno": "case_number",
    "registration_year": "year",
    "miningcase": "mining_case"
}

UNCTAD_MAPPING = {
    "country_clean": "country_clean",
    "CASE NO.": "case_number",
    "YEAR OF INITIATION": "year",
    "MINING CASE": "mining_case"
}

PCA_MAPPING = {
    "country_clean": "country_clean",
    "Case number": "case_number",
    "year": "year",
    "MINING CASE": "mining_case"
}

In [51]:
# Keep only rows from UNCTAD that aren't in ICSID and only rows from PCA that aren't in UNCTAD, then merge all three datasets together for analysis.
# Apply column renaming defined above

# ICSID: keep all rows
ICSID = df_clean.rename(columns=ICSID_MAPPING)[
    ["country_clean", "case_number", "year", "mining_case"]
]

# Keep only rows from UNCTAD that aren't in ICSID
UNCTAD_unique = UNCTAD_FULL[UNCTAD_FULL["IN ICSID DATA"] == "No"].rename(columns=UNCTAD_MAPPING)[
    ["country_clean", "case_number", "year", "mining_case"]
]

# Keep only rows from PCA that aren't in UNCTAD
PCA_unique = PCA_DATA[PCA_DATA["IN UNCTAD"] == "No"].rename(columns=PCA_MAPPING)[
    ["country_clean", "case_number", "year", "mining_case"]
]

In [52]:
# Combine all three datasets into one.
# Full mining cases
df_MINING_CASES = pd.concat(
    [ICSID, UNCTAD_unique, PCA_unique],
    ignore_index=True
)

print(f"Total amount of unique cases: {df_MINING_CASES.shape[0]}.")

# Save analysis dataset
df_MINING_CASES.to_csv(r"../data/processed/MINING_CASES_202606.csv", index=False)

Total amount of unique cases: 1839.


### Adding in Developed and Devloping economy defintions using the UNCTAD dataset
Retreived from https://unctadstat.unctad.org/EN/Classifications.html#:~:text=The%20developed%20economies%20broadly%20comprise,as%20Australia%20and%20New%20Zealand

In [53]:
# Only obtain economy classifications from each country.
df_economy_raw = pd.read_csv(r"../data/raw/UNCTAD_Dim_Countries_Hierarchy_UnctadStat_All_Flat.xls")
df_economy = df_economy_raw.copy()

df_economy_classification = df_economy[(df_economy["Parent_Code"] == 1400) | (df_economy["Parent_Code"] == 1500)]
df_economy_classification.to_csv(r"../data/interim/UNCTAD_economy_classification.csv")

In [54]:
# Standardise country names across both datasets
df_MINING_CASES['country_clean'] = df_MINING_CASES['country_clean'].str.strip().str.title()
df_economy_classification['Country_Label'] = df_economy_classification['Child_Label'].str.strip().str.title()

# Merge dataframes to include economic classification of the country
df_MINING_CASES = df_MINING_CASES.merge(
    df_economy_classification[["Child_Label", "Parent_Label"]],
    left_on="country_clean",
    right_on="Child_Label",
    how="left"
)

df_MINING_CASES.drop(columns=["Child_Label"], inplace=True)
df_MINING_CASES.rename(columns={"Parent_Label": "economy_classification"}, inplace=True)

df_MINING_CASES.head()

,country_clean,case_number,year,mining_case,economy_classification
0,Lebanon,ARB/26/22,2026,No,Developing economies
1,Ireland,ARB/26/21,2026,Yes,Developed economies
2,Colombia,ARB/26/20,2026,No,Developing economies
3,Mexico,ARB/26/19,2026,No,Developing economies
4,Australia,ARB/26/18,2026,No,Developed economies


In [56]:
# Check for missing labels
missing_cases = df_MINING_CASES["economy_classification"].isna()
print(f"Total number of cases missing a classification: {missing_cases.sum()}")

Total number of cases missing a classification: 207


In [58]:
# Missing classifications mainly due to string discrepancies.  Manually update economy classifications.
from utils.constant_variables import ECONOMY_MAP_MANUAL

df_MINING_CASES.loc[
    missing_cases,
    "economy_classification"
] = (
    df_MINING_CASES.loc[
        missing_cases,
        "country_clean"].map(ECONOMY_MAP_MANUAL)
)

missing_cases_verify = df_MINING_CASES["economy_classification"].isna()
# Verify all classifications given
print(f"Total number of cases missing a classification: {missing_cases_verify.sum()}")

Total number of cases missing a classification: 0


In [59]:
# Remove string "economies" from entry
df_MINING_CASES["economy_classification"] = ( df_MINING_CASES["economy_classification"].str.replace("economies", "", regex=False) )

# Remove leading/trailing whitespace from ALL string columns
df_MINING_CASES = df_MINING_CASES.apply(
    lambda col: col.str.strip() if col.dtype == "str" else col
)

# Check entries
df_MINING_CASES["economy_classification"].unique()

<StringArray>
['Developing', 'Developed']
Length: 2, dtype: str